# 📘 Day 4 — Text Preprocessing, Chunking & Keyword-Based Retrieval
### Course: Python for GenAI & Data Analysis | Date: 6 April 2026

---

## 🎯 Learning Objectives

By the end of this notebook, you will:
- ✅ Understand and implement **text preprocessing pipelines** from scratch
- ✅ Apply **5 different chunking strategies** and know when to use each
- ✅ Build a **keyword-based retrieval system** using TF-IDF and BM25
- ✅ Assemble a **mini RAG pipeline** connecting all three components
- ✅ Visualize and compare retrieval scores

---

## 📚 Table of Contents
1. [Setup & Imports](#1-setup)
2. [Text Preprocessing Techniques](#2-preprocessing)
3. [Document Chunking Strategies](#3-chunking)
4. [Keyword-Based Retrieval (TF-IDF & BM25)](#4-retrieval)
5. [End-to-End Mini RAG Pipeline](#5-rag)
6. [Visualization & Analysis](#6-viz)
7. [Summary & Day 5 Preview](#7-summary)

---
## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
# ─── Core Libraries ───────────────────────────────────────────────────────────
import re
import math
import string
import numpy as np
import collections
from collections import Counter, defaultdict
from typing import List, Tuple, Dict

# ─── Scikit-Learn ─────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ─── Visualization ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─── Optional: BM25 (install if available) ────────────────────────────────────
try:
    from rank_bm25 import BM25Okapi
    BM25_AVAILABLE = True
    print("✅ rank_bm25 loaded")
except ImportError:
    BM25_AVAILABLE = False
    print("ℹ️  rank_bm25 not installed — using built-in BM25 implementation")

# ─── NLTK (optional, graceful fallback) ───────────────────────────────────────
try:
    import nltk
    from nltk.tokenize import word_tokenize, sent_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer, PorterStemmer
    NLTK_AVAILABLE = True
    print("✅ NLTK loaded")
except Exception:
    NLTK_AVAILABLE = False
    print("ℹ️  NLTK not available — using built-in implementations")

print("\n🚀 All imports ready!")

In [ ]:
# ─── Built-in Fallbacks (no external downloads needed) ────────────────────────

# English stop words (common subset)
STOP_WORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'by','from','up','about','into','through','during','is','are','was',
    'were','be','been','being','have','has','had','do','does','did','will',
    'would','could','should','may','might','shall','can','need','dare',
    'ought','used','it','its','this','that','these','those','i','you','he',
    'she','we','they','what','which','who','whom','not','no','so','if',
    'as','than','then','there','their','they','our','your','his','her',
    'my','me','him','us','them','each','few','more','most','other','some',
    'such','any','only','same','also','just','because','while','how','all'
}

# Simple rule-based sentence splitter (fallback)
def simple_sent_tokenize(text: str) -> List[str]:
    """Split text into sentences using punctuation heuristics."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]

# Simple suffix-based stemmer (fallback)
def simple_stem(word: str) -> str:
    """Crude suffix stripping stemmer."""
    suffixes = ['ing', 'tion', 'ness', 'ment', 'ly', 'er', 'ed', 'es', 's']
    for suffix in suffixes:
        if word.endswith(suffix) and len(word) - len(suffix) > 3:
            return word[:-len(suffix)]
    return word

print("✅ Fallback utilities defined")
print(f"Stop words count: {len(STOP_WORDS)}")

---
## 2. Text Preprocessing Techniques <a id='2-preprocessing'></a>

> **Why does preprocessing matter?**  
> Raw text from the real world is messy — it contains HTML tags, inconsistent casing, special characters, and meaningless filler words. Preprocessing transforms noisy text into clean, normalized input that models and retrieval algorithms can work with effectively.

### The 7 Steps of Text Preprocessing:
```
Raw Text → Lowercase → Remove HTML → Remove Special Chars → 
Normalize Whitespace → Tokenize → Remove Stop Words → Stem/Lemmatize
```

### 2.1 Step 1 — Lowercasing

Ensures `Python`, `python`, and `PYTHON` are treated as the same token.

In [ ]:
sample = "The Quick Brown Fox Jumps Over The LAZY Dog! It's Running FAST."

# Step 1: Lowercase
lowercased = sample.lower()
print("Original: ", sample)
print("Lowercased:", lowercased)

# Why it matters — token counting example
from collections import Counter
raw_tokens = sample.split()
lower_tokens = lowercased.split()

print(f"\n📊 Unique tokens BEFORE lowercase: {len(set(raw_tokens))}")
print(f"📊 Unique tokens AFTER  lowercase: {len(set(lower_tokens))}")
print("→ Fewer unique tokens = smaller vocabulary = more efficient processing")

### 2.2 Step 2 — Removing HTML Tags

In [ ]:
html_text = """
<html>
  <body>
    <h1>Introduction to <b>Machine Learning</b></h1>
    <p>Machine learning is a <em>subset</em> of artificial intelligence.
    It enables systems to <strong>learn from data</strong> automatically.</p>
    <a href='https://example.com'>Learn more</a>
  </body>
</html>
"""

# Method 1: Simple regex
def remove_html_regex(text: str) -> str:
    return re.sub(r'<[^>]+>', ' ', text)

clean = remove_html_regex(html_text)
clean = re.sub(r'\s+', ' ', clean).strip()

print("RAW HTML (first 200 chars):")
print(html_text[:200])
print("\n" + "─"*60)
print("\nAFTER HTML removal:")
print(clean)

# Also handle HTML entities
html_entities = {
    '&amp;': '&', '&lt;': '<', '&gt;': '>', 
    '&nbsp;': ' ', '&quot;': '"', '&#39;': "'"
}

def decode_html_entities(text: str) -> str:
    for entity, char in html_entities.items():
        text = text.replace(entity, char)
    return text

test_entity = "Price: 10 &amp; 20, &lt;sale&gt; &quot;now&quot;"
print("\nHTML entities example:")
print(f"Before: {test_entity}")
print(f"After:  {decode_html_entities(test_entity)}")

### 2.3 Step 3 — Removing Special Characters & Punctuation

In [ ]:
messy_text = "Hello, World! This is NLP... it's #amazing @2024. $100 off! 50% discount??"

# Method 1: Remove everything except letters, digits, spaces
alphanumeric_only = re.sub(r'[^a-zA-Z0-9\s]', '', messy_text)

# Method 2: Remove punctuation but keep apostrophes (for contractions)
keep_apostrophe = re.sub(r"[^\w\s']", '', messy_text)

# Method 3: Using string.punctuation
no_punct = messy_text.translate(str.maketrans('', '', string.punctuation))

print(f"Original:         {messy_text}")
print(f"Alphanumeric:     {alphanumeric_only}")
print(f"Keep apostrophe:  {keep_apostrophe}")
print(f"No punctuation:   {no_punct}")

# ─── Practical Note ───────────────────────────────────────────────────────────
print("\n💡 Tip: Choice depends on use case:")
print("   - For keyword search → alphanumeric_only")
print("   - For sentiment analysis → keep punctuation (! ? affect meaning)")
print("   - For embeddings → usually keep more, let model handle it")

### 2.4 Step 4 — Whitespace Normalization

In [ ]:
noisy_spaces = "  This   has   extra   spaces   \n\n\t and   newlines  \n  everywhere  "

# Collapse all whitespace (spaces, tabs, newlines) to single space
normalized = re.sub(r'\s+', ' ', noisy_spaces).strip()

print(f"Original  (repr): {repr(noisy_spaces[:60])}...")
print(f"Normalized:       {normalized}")

# Show character counts
print(f"\nOriginal length:   {len(noisy_spaces)}")
print(f"Normalized length: {len(normalized)}")
print(f"Characters saved:  {len(noisy_spaces) - len(normalized)}")

### 2.5 Step 5 — Tokenization

Tokenization splits text into individual units (tokens). The granularity depends on your task.

In [ ]:
text = "Dr. Smith can't believe it! He said, 'NLP is amazing.' Really? Yes, it is."

# ── Word Tokenization ─────────────────────────────────────────────────────────

# Method 1: Simple split (naive)
naive_tokens = text.split()

# Method 2: Regex-based (better — handles punctuation)
regex_tokens = re.findall(r"\b[\w']+\b", text)

# Method 3: NLTK word_tokenize (best — if available)
if NLTK_AVAILABLE:
    nltk_tokens = word_tokenize(text)
else:
    nltk_tokens = regex_tokens  # fallback

print("Text:", text)
print(f"\n1. Naive split ({len(naive_tokens)} tokens):")
print(naive_tokens)
print(f"\n2. Regex tokens ({len(regex_tokens)} tokens):")
print(regex_tokens)
print(f"\n3. NLTK tokens ({len(nltk_tokens)} tokens):")
print(nltk_tokens)

# ── Sentence Tokenization ─────────────────────────────────────────────────────
print("\n" + "─"*60)
print("\nSentence Tokenization:")
sentences = simple_sent_tokenize(text)
for i, s in enumerate(sentences, 1):
    print(f"  Sentence {i}: {s}")

In [ ]:
# ─── Subword Tokenization (used by LLMs) ──────────────────────────────────────
# LLMs like GPT use Byte Pair Encoding (BPE) — let's simulate the concept

print("📚 Subword Tokenization (used in GPT, BERT, etc.)")
print("─" * 55)
print()

# Demonstrate how LLM tokenizers split words
examples = [
    ("unhappiness",  ["un", "##happi", "##ness"]),      # BERT-style
    ("preprocessing", ["pre", "process", "ing"]),        # BPE-style
    ("ChatGPT",       ["Chat", "G", "PT"]),
    ("2024",          ["20", "24"]),
    ("unbelievable",  ["un", "believ", "able"]),
]

print(f"{'Word':<20} {'Subword Tokens':<35} {'# Tokens'}") 
print("─" * 65)
for word, tokens in examples:
    print(f"{word:<20} {str(tokens):<35} {len(tokens)}")

print()
print("💡 Key Insight: LLMs never see whole words — they see SUBWORDS (tokens).")
print("   - 'token count' in LLM pricing refers to these subword tokens")
print("   - Rare/new words get split into more tokens (less efficient)")
print("   - Common words are usually single tokens")

### 2.6 Step 6 — Stop Word Removal

In [ ]:
text = "the quick brown fox jumps over the lazy dog and it is very fast"
tokens = text.split()

# Remove stop words
filtered = [t for t in tokens if t not in STOP_WORDS]

print(f"Original tokens ({len(tokens)}): {tokens}")
print(f"\nFiltered tokens ({len(filtered)}): {filtered}")
print(f"\nRemoved {len(tokens) - len(filtered)} stop words ({(len(tokens)-len(filtered))/len(tokens)*100:.1f}% reduction)")

# Visualize which words were removed
print("\nLegend: [STOP] = removed, [KEEP] = kept")
for t in tokens:
    tag = "[STOP]" if t in STOP_WORDS else "[KEEP]"
    print(f"  {tag}  {t}")

print()
print("⚠️  When NOT to remove stop words:")
print("   • Using transformer embeddings (BERT, OpenAI) — keep everything")
print("   • Sentiment analysis — 'not good' vs 'good' (negation matters!)")
print("   • Named entity recognition — stop words provide context")

### 2.7 Step 7 — Stemming vs Lemmatization

Both reduce words to their root form, but with different approaches and accuracy levels.

In [ ]:
# ─── Stemming ─────────────────────────────────────────────────────────────────
print("🔪 STEMMING (crude suffix stripping)")
print("─" * 55)

test_words = [
    "running", "runs", "runner",
    "studies", "studied", "studying",
    "better", "happiness", "beautiful",
    "organization", "organizations",
    "generative", "generates", "generation"
]

# Our built-in simple stemmer
stemmed = [(w, simple_stem(w)) for w in test_words]

print(f"{'Original':<20} {'Stemmed':<20} {'Changed?'}")
print("─" * 55)
for orig, stem in stemmed:
    changed = "✅" if orig != stem else "—"
    print(f"{orig:<20} {stem:<20} {changed}")

print()
print("⚠️  Problems with stemming:")
print("   'studies' → 'studi'  (not a real word!)")
print("   'better'  → 'better' (missed — means 'good')")

In [ ]:
# ─── Lemmatization vs Stemming Side-by-Side ────────────────────────────────────
print("📖 STEMMING vs LEMMATIZATION Comparison")
print("─" * 70)

# Hardcoded lemma examples (what a lemmatizer WOULD return)
comparison = [
    ("running",      "run",          "runn"),
    ("studies",      "study",        "studi"),       # stemming breaks this!
    ("better",       "good",         "better"),      # stemming misses this!
    ("was",          "be",           "was"),         # stemming can't handle irregular
    ("geese",        "goose",        "gees"),        # irregular plural
    ("computing",    "compute",      "comput"),
    ("generative",   "generative",   "generat"),
    ("organizations","organization", "organ"),       # over-stemmed!
    ("beautiful",    "beautiful",    "beauti"),
    ("happily",      "happily",      "happili"),
]

print(f"{'Word':<18} {'Lemma (correct)':<22} {'Stem (crude)':<18} {'Notes'}")
print("─" * 80)
for word, lemma, stem in comparison:
    note = ""
    if lemma != stem and stem not in ["be", "good", "goose"]:
        note = "← stem breaks here" if len(stem) < len(lemma) - 2 else ""
    print(f"{word:<18} {lemma:<22} {stem:<18} {note}")

print()
print("📌 Rule of thumb:")
print("   Use STEMMING  → when speed matters more than accuracy (search engines, large datasets)")
print("   Use LEMMATIZE → when accuracy matters (chatbots, NLU, RAG pipelines)")

### 2.8 Complete Preprocessing Pipeline

In [ ]:
class TextPreprocessor:
    """
    A complete, configurable text preprocessing pipeline.
    
    Parameters:
    -----------
    lowercase      : Convert to lowercase
    remove_html    : Strip HTML tags
    remove_special : Remove special characters
    remove_stops   : Remove stop words
    stem           : Apply stemming
    min_token_len  : Minimum length for keeping a token
    """
    
    def __init__(
        self,
        lowercase: bool = True,
        remove_html: bool = True,
        remove_special: bool = True,
        remove_stops: bool = True,
        stem: bool = False,
        min_token_len: int = 2
    ):
        self.lowercase = lowercase
        self.remove_html = remove_html
        self.remove_special = remove_special
        self.remove_stops = remove_stops
        self.stem = stem
        self.min_token_len = min_token_len
        self._steps_log = []
    
    def process(self, text: str, verbose: bool = False) -> str:
        """Apply the full preprocessing pipeline."""
        self._steps_log = [('raw', text)]
        
        if self.lowercase:
            text = text.lower()
            self._steps_log.append(('lowercase', text))
        
        if self.remove_html:
            text = re.sub(r'<[^>]+>', ' ', text)
            text = decode_html_entities(text)
            self._steps_log.append(('remove_html', text))
        
        if self.remove_special:
            text = re.sub(r'[^a-z0-9\s]', '', text)
            self._steps_log.append(('remove_special', text))
        
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        self._steps_log.append(('normalize_whitespace', text))
        
        # Tokenize
        tokens = text.split()
        self._steps_log.append(('tokenize', f"{len(tokens)} tokens"))
        
        # Filter by length
        tokens = [t for t in tokens if len(t) >= self.min_token_len]
        
        if self.remove_stops:
            tokens = [t for t in tokens if t not in STOP_WORDS]
            self._steps_log.append(('remove_stops', f"{len(tokens)} tokens"))
        
        if self.stem:
            tokens = [simple_stem(t) for t in tokens]
            self._steps_log.append(('stem', str(tokens[:5]) + '...'))
        
        result = ' '.join(tokens)
        
        if verbose:
            self._print_steps()
        
        return result
    
    def process_batch(self, texts: List[str]) -> List[str]:
        """Process a list of texts."""
        return [self.process(t) for t in texts]
    
    def _print_steps(self):
        print("📋 Preprocessing Steps:")
        print("─" * 60)
        for step, value in self._steps_log:
            val_str = str(value)[:70] + '...' if len(str(value)) > 70 else str(value)
            print(f"  [{step:>25}]  {val_str}")

print("✅ TextPreprocessor class defined")

In [ ]:
# ─── Demo the full pipeline ────────────────────────────────────────────────────
raw_text = """<h2>Introduction to <b>Generative AI</b></h2>
<p>Generative AI models, like GPT-4 &amp; Claude, are trained on massive datasets!
They can generate text, images, code, and much more. It's a revolutionary technology
that's transforming industries across the world in 2024.</p>"""

preprocessor = TextPreprocessor(
    lowercase=True,
    remove_html=True,
    remove_special=True,
    remove_stops=True,
    stem=False,
    min_token_len=2
)

result = preprocessor.process(raw_text, verbose=True)
print(f"\n✨ FINAL OUTPUT: '{result}'")

In [ ]:
# ─── Compare with and without stop words for retrieval ──────────────────────
doc = "Machine learning is a type of artificial intelligence that allows computers to learn without being explicitly programmed"

pp_with_stops = TextPreprocessor(remove_stops=False, stem=False)
pp_no_stops   = TextPreprocessor(remove_stops=True,  stem=False)
pp_stemmed    = TextPreprocessor(remove_stops=True,  stem=True)

v1 = pp_with_stops.process(doc)
v2 = pp_no_stops.process(doc)
v3 = pp_stemmed.process(doc)

print("Original:")
print(f"  {doc}")
print(f"\n1. With stops  ({len(v1.split())} tokens): {v1}")
print(f"2. No stops    ({len(v2.split())} tokens): {v2}")
print(f"3. Stemmed     ({len(v3.split())} tokens): {v3}")

---
## 3. Document Chunking Strategies <a id='3-chunking'></a>

> **Why chunk documents?**  
> LLMs have limited context windows. A 50-page document cannot fit in one API call. Chunking splits documents into smaller, semantically meaningful pieces that can be:
> - **Indexed** efficiently in a vector database  
> - **Retrieved** precisely (only relevant chunks, not the whole doc)  
> - **Fed** to an LLM within its context window limit

```
                    Document
                       │
          ┌────────────┴────────────┐
          ▼            ▼            ▼
       Chunk 1      Chunk 2      Chunk 3   ← Indexed in vector store
          │
      (retrieved for query Q)
          │
          ▼
      LLM Context Window  →  Answer
```

In [ ]:
# ─── Sample Document for All Chunking Experiments ─────────────────────────────

SAMPLE_DOC = """Artificial intelligence (AI) is intelligence demonstrated by machines, 
as opposed to the natural intelligence displayed by animals including humans. 
AI research has been defined as the field of study of intelligent agents, which refers to any 
system that perceives its environment and takes actions that maximize its chance of achieving 
its goals.

Machine learning is a subset of AI that gives computers the ability to learn without being 
explicitly programmed. It focuses on developing computer programs that can access data and 
use it to learn for themselves. The process begins with observations or data, such as examples, 
direct experience, or instruction, so that computers can look for patterns in data and make 
better decisions in the future.

Deep learning is a subset of machine learning that uses neural networks with many layers. 
These deep neural networks have led to dramatic advances in computer vision, natural language 
processing, and speech recognition. Deep learning models learn representations of data with 
multiple levels of abstraction.

Natural language processing (NLP) is a subfield of linguistics, computer science, and 
artificial intelligence concerned with the interactions between computers and human language, 
in particular how to program computers to process and analyze large amounts of natural 
language data. The goal is a computer capable of understanding the contents of documents, 
including the contextual nuances of the language within them.

Retrieval Augmented Generation (RAG) is a technique that combines retrieval systems with 
generative models. Instead of relying solely on the parametric knowledge stored in model 
weights, RAG systems retrieve relevant documents from an external knowledge base and use 
them as context when generating responses. This allows models to access up-to-date information 
and reduces hallucination."""

print(f"Document length: {len(SAMPLE_DOC)} characters")
print(f"Word count: {len(SAMPLE_DOC.split())} words")
print(f"Estimated tokens (÷4): ~{len(SAMPLE_DOC)//4} tokens")
print(f"\nFirst 200 chars: {SAMPLE_DOC[:200]}...")

### 3.1 Strategy 1 — Fixed-Size Chunking

In [ ]:
def fixed_size_chunk(
    text: str,
    chunk_size: int = 200,
    overlap: int = 30
) -> List[Dict]:
    """
    Split text into fixed-size character chunks with overlap.
    
    Parameters:
    -----------
    text       : Input text
    chunk_size : Number of characters per chunk
    overlap    : Number of overlapping characters between consecutive chunks
    
    Returns:
    --------
    List of dicts with keys: 'id', 'text', 'start', 'end', 'size'
    """
    chunks = []
    start = 0
    chunk_id = 0
    
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk_text = text[start:end]
        
        chunks.append({
            'id': chunk_id,
            'text': chunk_text,
            'start': start,
            'end': end,
            'size': len(chunk_text)
        })
        
        start = end - overlap  # Move forward, but overlap keeps context
        chunk_id += 1
        
        if start >= len(text):
            break
    
    return chunks


# ─── Demo ─────────────────────────────────────────────────────────────────────
fixed_chunks = fixed_size_chunk(SAMPLE_DOC, chunk_size=300, overlap=50)

print(f"Fixed-Size Chunking (size=300, overlap=50):")
print(f"Total chunks: {len(fixed_chunks)}")
print()

for chunk in fixed_chunks[:3]:  # show first 3
    print(f"─── Chunk {chunk['id']} (chars {chunk['start']}–{chunk['end']}, size={chunk['size']}) ───")
    print(chunk['text'][:120] + '...' if len(chunk['text']) > 120 else chunk['text'])
    print()

print("⚠️  Notice: Chunks may split mid-sentence!")
print(f"   e.g., chunk 0 ends at: '...{fixed_chunks[0]['text'][-30:]}'")

### 3.2 Strategy 2 — Sentence-Based Chunking

In [ ]:
def sentence_chunk(
    text: str,
    sentences_per_chunk: int = 3,
    overlap_sentences: int = 1
) -> List[Dict]:
    """
    Split text into chunks, each containing N sentences.
    
    Parameters:
    -----------
    text                 : Input text
    sentences_per_chunk  : Number of sentences per chunk
    overlap_sentences    : Number of sentences shared between consecutive chunks
    """
    # Use NLTK if available, otherwise our simple splitter
    if NLTK_AVAILABLE:
        sentences = sent_tokenize(text)
    else:
        sentences = simple_sent_tokenize(text)
    
    chunks = []
    step = max(1, sentences_per_chunk - overlap_sentences)
    
    for i in range(0, len(sentences), step):
        chunk_sents = sentences[i : i + sentences_per_chunk]
        if chunk_sents:
            chunks.append({
                'id': len(chunks),
                'text': ' '.join(chunk_sents),
                'sentence_range': (i, i + len(chunk_sents) - 1),
                'size': len(' '.join(chunk_sents)),
                'num_sentences': len(chunk_sents)
            })
    
    return chunks


# ─── Demo ─────────────────────────────────────────────────────────────────────
sent_chunks = sentence_chunk(SAMPLE_DOC, sentences_per_chunk=3, overlap_sentences=1)

print(f"Sentence-Based Chunking (3 sentences/chunk, 1 sentence overlap):")
print(f"Total chunks: {len(sent_chunks)}")
print()

for chunk in sent_chunks[:3]:
    print(f"─── Chunk {chunk['id']} (sentences {chunk['sentence_range']}, size={chunk['size']}) ───")
    print(chunk['text'][:200] + '...' if len(chunk['text']) > 200 else chunk['text'])
    print()

print("✅ Notice: Every chunk ends at a sentence boundary!")

### 3.3 Strategy 3 — Paragraph-Based Chunking

In [ ]:
def paragraph_chunk(
    text: str,
    min_chars: int = 50,
    max_chars: int = None
) -> List[Dict]:
    """
    Split text at paragraph breaks (double newlines).
    
    Parameters:
    -----------
    text      : Input text
    min_chars : Minimum characters for a chunk to be kept
    max_chars : Optional maximum — longer paragraphs will be further split
    """
    # Split on double newlines (paragraph separator)
    paragraphs = re.split(r'\n\n+', text)
    chunks = []
    
    for para in paragraphs:
        para = re.sub(r'\s+', ' ', para).strip()
        
        if len(para) < min_chars:
            continue  # Skip tiny fragments
        
        if max_chars and len(para) > max_chars:
            # Further split large paragraphs at sentences
            sub_chunks = sentence_chunk(para, sentences_per_chunk=2)
            for sc in sub_chunks:
                chunks.append({
                    'id': len(chunks),
                    'text': sc['text'],
                    'size': sc['size'],
                    'source': 'paragraph_split'
                })
        else:
            chunks.append({
                'id': len(chunks),
                'text': para,
                'size': len(para),
                'source': 'paragraph'
            })
    
    return chunks


# ─── Demo ─────────────────────────────────────────────────────────────────────
para_chunks = paragraph_chunk(SAMPLE_DOC)

print(f"Paragraph-Based Chunking:")
print(f"Total chunks: {len(para_chunks)}")
print(f"Sizes: {[c['size'] for c in para_chunks]}")
print()

for chunk in para_chunks:
    print(f"─── Chunk {chunk['id']} ({chunk['size']} chars) ───")
    print(chunk['text'][:180] + '...' if len(chunk['text']) > 180 else chunk['text'])
    print()

### 3.4 Strategy 4 — Recursive Chunking (LangChain Default)

In [ ]:
def recursive_chunk(
    text: str,
    chunk_size: int = 400,
    overlap: int = 60,
    separators: List[str] = None
) -> List[Dict]:
    """
    Recursively split text using progressively finer separators.
    This is the approach used by LangChain's RecursiveCharacterTextSplitter.
    
    Separator priority:
    1. Double newline (paragraph)  →  best natural break
    2. Single newline              →  line break  
    3. Period + space              →  sentence boundary
    4. Space                       →  word boundary
    5. Empty string                →  character boundary (last resort)
    """
    if separators is None:
        separators = ['\n\n', '\n', '. ', ' ', '']
    
    def _split(text: str, seps: List[str]) -> List[str]:
        if not seps:
            # Base case: split into char chunks
            return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size-overlap)]
        
        sep = seps[0]
        remaining_seps = seps[1:]
        
        if sep not in text:
            return _split(text, remaining_seps)
        
        parts = text.split(sep)
        result = []
        current = ""
        
        for part in parts:
            part = part.strip()
            if not part:
                continue
            
            candidate = (current + sep + part).strip() if current else part
            
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    result.append(current)
                
                if len(part) > chunk_size:
                    # Part itself is too large → recurse with finer separator
                    sub_parts = _split(part, remaining_seps)
                    result.extend(sub_parts[:-1])
                    current = sub_parts[-1] if sub_parts else ""
                else:
                    current = part
        
        if current:
            result.append(current)
        
        return result
    
    raw_chunks = _split(text, separators)
    
    # Add overlap between consecutive chunks
    result = []
    for i, chunk in enumerate(raw_chunks):
        if i > 0 and overlap > 0:
            prev = raw_chunks[i-1]
            tail = prev[-overlap:] if len(prev) > overlap else prev
            chunk = tail + ' ' + chunk
        
        result.append({
            'id': i,
            'text': chunk.strip(),
            'size': len(chunk.strip()),
            'strategy': 'recursive'
        })
    
    return result


# ─── Demo ─────────────────────────────────────────────────────────────────────
rec_chunks = recursive_chunk(SAMPLE_DOC, chunk_size=400, overlap=50)

print(f"Recursive Chunking (size=400, overlap=50):")
print(f"Total chunks: {len(rec_chunks)}")
print()

for chunk in rec_chunks[:3]:
    print(f"─── Chunk {chunk['id']} ({chunk['size']} chars) ───")
    print(chunk['text'][:200] + '...' if len(chunk['text']) > 200 else chunk['text'])
    print()

### 3.5 Chunking Strategy Comparison

In [ ]:
# ─── Compare all strategies on the same document ──────────────────────────────
strategies = {
    'Fixed-Size\n(size=300)':   fixed_size_chunk(SAMPLE_DOC, chunk_size=300, overlap=50),
    'Sentence-Based\n(3 sents)': sentence_chunk(SAMPLE_DOC, sentences_per_chunk=3),
    'Paragraph\nBased':          paragraph_chunk(SAMPLE_DOC),
    'Recursive\n(size=400)':     recursive_chunk(SAMPLE_DOC, chunk_size=400, overlap=50),
}

print(f"{'Strategy':<28} {'# Chunks':>10} {'Avg Size':>10} {'Min':>8} {'Max':>8}")
print("─" * 70)

for name, chunks in strategies.items():
    sizes = [c['size'] for c in chunks]
    name_clean = name.replace('\n', ' ')
    print(f"{name_clean:<28} {len(chunks):>10} {np.mean(sizes):>10.0f} {min(sizes):>8} {max(sizes):>8}")

# ─── Visualization ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Chunking Strategy Comparison', fontsize=14, fontweight='bold')

# Plot 1: Number of chunks
names = [n.replace('\n', '\n') for n in strategies.keys()]
n_chunks = [len(v) for v in strategies.values()]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

axes[0].bar(names, n_chunks, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Number of Chunks Produced')
axes[0].set_ylabel('Count')
for i, v in enumerate(n_chunks):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# Plot 2: Chunk size distributions
for i, (name, chunks) in enumerate(strategies.items()):
    sizes = [c['size'] for c in chunks]
    axes[1].scatter([i]*len(sizes), sizes, color=colors[i], alpha=0.7, s=80, zorder=3)
    axes[1].plot([i-0.2, i+0.2], [np.mean(sizes)]*2, color=colors[i], linewidth=3)

axes[1].set_title('Chunk Size Distribution (dot = chunk, line = mean)')
axes[1].set_ylabel('Characters')
axes[1].set_xticks(range(len(strategies)))
axes[1].set_xticklabels(names)

plt.tight_layout()
plt.savefig('chunking_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("📊 Chart saved as 'chunking_comparison.png'")

---
## 4. Keyword-Based Retrieval (TF-IDF & BM25) <a id='4-retrieval'></a>

> **Keyword retrieval** finds documents by matching and ranking terms from a query against an indexed corpus.  
> Unlike semantic search (vectors/embeddings), it is **exact**, **fast**, and **interpretable**.

### Key Concept: TF-IDF

$$\text{TF-IDF}(t, d, D) = \underbrace{\frac{f_{t,d}}{\sum_k f_{k,d}}}_{\text{TF}} \times \underbrace{\log\frac{|D|}{|\{d:t\in d\}|}}_{\text{IDF}}$$

- **TF**: How often does term `t` appear in document `d`?  
- **IDF**: How rare is term `t` across ALL documents?

In [ ]:
# ─── Build TF-IDF from Scratch ────────────────────────────────────────────────

class TFIDFFromScratch:
    """
    TF-IDF implementation from scratch — educational version.
    Helps you understand exactly what sklearn's TfidfVectorizer does internally.
    """
    
    def __init__(self):
        self.vocabulary = {}
        self.idf_scores = {}
        self.N = 0  # number of documents
    
    def _tokenize(self, text: str) -> List[str]:
        return re.findall(r'\b[a-z]+\b', text.lower())
    
    def fit(self, corpus: List[str]):
        """Learn vocabulary and IDF scores from corpus."""
        self.N = len(corpus)
        
        # Build vocabulary and document frequencies
        df = defaultdict(int)  # document frequency per term
        
        for doc in corpus:
            terms = set(self._tokenize(doc))  # unique terms per doc
            for term in terms:
                df[term] += 1
        
        # Build vocabulary (sorted for consistency)
        self.vocabulary = {term: idx for idx, term in enumerate(sorted(df.keys()))}
        
        # Calculate IDF: log(N / df(t)) + 1 (sklearn's smooth IDF)
        self.idf_scores = {
            term: math.log(self.N / count) + 1
            for term, count in df.items()
        }
        
        return self
    
    def _tf(self, tokens: List[str]) -> Dict[str, float]:
        """Calculate term frequency for a list of tokens."""
        counts = Counter(tokens)
        total = len(tokens)
        return {term: count / total for term, count in counts.items()}
    
    def transform(self, documents: List[str]) -> np.ndarray:
        """Convert documents to TF-IDF matrix."""
        matrix = np.zeros((len(documents), len(self.vocabulary)))
        
        for i, doc in enumerate(documents):
            tokens = self._tokenize(doc)
            tf = self._tf(tokens)
            
            for term, tf_score in tf.items():
                if term in self.vocabulary:
                    j = self.vocabulary[term]
                    idf = self.idf_scores.get(term, 0)
                    matrix[i, j] = tf_score * idf
        
        # L2 normalize each row
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1
        return matrix / norms
    
    def fit_transform(self, corpus: List[str]) -> np.ndarray:
        return self.fit(corpus).transform(corpus)


# ─── Demo with small corpus ───────────────────────────────────────────────────
mini_corpus = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks with many layers",
    "natural language processing analyzes text data",
    "retrieval systems find relevant documents for queries",
    "text preprocessing cleans and normalizes raw text"
]

tfidf_scratch = TFIDFFromScratch()
matrix = tfidf_scratch.fit_transform(mini_corpus)

print(f"TF-IDF Matrix shape: {matrix.shape}")
print(f"(documents × vocabulary terms)")
print(f"\nVocabulary size: {len(tfidf_scratch.vocabulary)} terms")
print(f"\nTop IDF scores (rarest = most informative terms):")
idf_sorted = sorted(tfidf_scratch.idf_scores.items(), key=lambda x: x[1], reverse=True)
for term, score in idf_sorted[:10]:
    print(f"  {term:<20} IDF = {score:.3f}")

In [ ]:
# ─── TF-IDF with Sklearn (production-ready) ───────────────────────────────────

# Use paragraph chunks from our document as the corpus
chunk_texts = [c['text'] for c in para_chunks]

print(f"Corpus: {len(chunk_texts)} chunks from sample document")
print()

# Fit TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=1000,       # Limit vocabulary size
    ngram_range=(1, 2),      # Include bigrams (e.g., "machine learning")
    stop_words='english',    # Remove English stop words
    min_df=1,                # Minimum document frequency
    sublinear_tf=True        # Apply log normalization to TF
)

tfidf_matrix = vectorizer.fit_transform(chunk_texts)
feature_names = vectorizer.get_feature_names_out()

print(f"TF-IDF Matrix: {tfidf_matrix.shape[0]} chunks × {tfidf_matrix.shape[1]} terms")
print(f"\nTop 20 features (vocabulary):")
print(feature_names[:20].tolist())

# Show top TF-IDF terms for each chunk
print("\n" + "─"*60)
print("Top 5 TF-IDF terms per chunk:")
print("─"*60)
for i, chunk in enumerate(chunk_texts):
    row = tfidf_matrix[i].toarray()[0]
    top_indices = np.argsort(row)[::-1][:5]
    top_terms = [(feature_names[j], round(row[j], 3)) for j in top_indices if row[j] > 0]
    print(f"Chunk {i}: {top_terms}")

In [ ]:
# ─── Retrieval with TF-IDF + Cosine Similarity ────────────────────────────────

def retrieve_tfidf(
    query: str,
    corpus: List[str],
    vectorizer: TfidfVectorizer,
    tfidf_matrix,
    top_k: int = 3
) -> List[Tuple[int, float, str]]:
    """
    Retrieve top-k relevant chunks for a query using TF-IDF cosine similarity.
    
    Returns: List of (rank, score, chunk_text) tuples
    """
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    return [
        (rank + 1, float(scores[idx]), corpus[idx])
        for rank, idx in enumerate(top_indices)
        if scores[idx] > 0
    ]


# ─── Test queries ─────────────────────────────────────────────────────────────
test_queries = [
    "What is machine learning?",
    "How do neural networks work?",
    "What is RAG and why is it useful?",
    "text preprocessing for NLP",
]

for query in test_queries:
    results = retrieve_tfidf(query, chunk_texts, vectorizer, tfidf_matrix, top_k=2)
    print(f"\n🔍 Query: '{query}'")
    print("─" * 60)
    if results:
        for rank, score, text in results:
            print(f"  Rank {rank} (score={score:.4f}):")
            print(f"    {text[:150]}..." if len(text) > 150 else f"    {text}")
    else:
        print("  No results found.")

### 4.2 BM25 — Best Match 25

BM25 improves TF-IDF with two key enhancements:
1. **Term Saturation**: More occurrences of a term add diminishing returns
2. **Document Length Normalization**: Penalizes unnecessarily long documents

$$\text{BM25}(t,d) = \text{IDF}(t) \cdot \frac{(k_1+1) \cdot f_{t,d}}{f_{t,d} + k_1 \cdot (1 - b + b \cdot \frac{|d|}{\text{avgdl}})}$$

- `k1` (default 1.5): Controls term saturation (higher = less saturation)
- `b` (default 0.75): Controls length normalization (1 = full, 0 = none)

In [ ]:
# ─── BM25 From Scratch ────────────────────────────────────────────────────────

class BM25FromScratch:
    """
    BM25 (Okapi BM25) implementation from scratch.
    Used by: Elasticsearch, Apache Solr, many production search engines.
    """
    
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1   # term saturation parameter
        self.b  = b    # length normalization parameter
        self.corpus_tokenized = []
        self.N = 0
        self.avgdl = 0.0
        self.df = {}
        self.idf = {}
    
    def _tokenize(self, text: str) -> List[str]:
        tokens = re.findall(r'\b[a-z]+\b', text.lower())
        return [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    
    def fit(self, corpus: List[str]):
        self.corpus_tokenized = [self._tokenize(doc) for doc in corpus]
        self.N = len(corpus)
        
        # Average document length
        lengths = [len(doc) for doc in self.corpus_tokenized]
        self.avgdl = sum(lengths) / self.N if self.N > 0 else 1
        
        # Document frequencies
        self.df = defaultdict(int)
        for doc_tokens in self.corpus_tokenized:
            for term in set(doc_tokens):
                self.df[term] += 1
        
        # IDF (BM25 uses a slightly different IDF formula than TF-IDF)
        self.idf = {
            term: math.log((self.N - df + 0.5) / (df + 0.5) + 1)
            for term, df in self.df.items()
        }
        
        return self
    
    def score(self, query: str, doc_idx: int) -> float:
        """Calculate BM25 score for a single query-document pair."""
        query_tokens = self._tokenize(query)
        doc_tokens = self.corpus_tokenized[doc_idx]
        doc_len = len(doc_tokens)
        doc_counter = Counter(doc_tokens)
        
        score = 0.0
        for term in query_tokens:
            if term not in self.idf:
                continue
            
            tf = doc_counter.get(term, 0)
            
            # BM25 term score with saturation and length normalization
            numerator   = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            
            score += self.idf[term] * (numerator / denominator)
        
        return score
    
    def get_scores(self, query: str) -> List[float]:
        return [self.score(query, i) for i in range(self.N)]
    
    def retrieve(self, query: str, top_k: int = 3, corpus: List[str] = None) -> List[Tuple]:
        scores = self.get_scores(query)
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for rank, idx in enumerate(top_indices):
            if scores[idx] > 0 and corpus:
                results.append((rank+1, round(scores[idx], 4), corpus[idx]))
        return results


# ─── Fit BM25 on our chunk corpus ─────────────────────────────────────────────
bm25 = BM25FromScratch(k1=1.5, b=0.75)
bm25.fit(chunk_texts)

print(f"BM25 fitted on {bm25.N} documents")
print(f"Average document length: {bm25.avgdl:.1f} tokens")
print(f"Vocabulary size: {len(bm25.df)} terms")
print()

# Test retrieval
for query in test_queries:
    results = bm25.retrieve(query, top_k=2, corpus=chunk_texts)
    print(f"🔍 Query: '{query}'")
    for rank, score, text in results:
        print(f"   Rank {rank} (BM25={score}): {text[:120]}...")
    print()

In [ ]:
# ─── TF-IDF vs BM25 Score Comparison ─────────────────────────────────────────

query = "What is machine learning and artificial intelligence?"

# TF-IDF scores
q_vec = vectorizer.transform([query])
tfidf_scores = cosine_similarity(q_vec, tfidf_matrix)[0]

# BM25 scores
bm25_scores = np.array(bm25.get_scores(query))

# Normalize BM25 scores to 0-1 for fair comparison
bm25_norm = bm25_scores / (bm25_scores.max() + 1e-9)

print(f"Query: '{query}'")
print()
print(f"{'Chunk':<8} {'TF-IDF Score':>14} {'BM25 (norm)':>13} {'Rank TF-IDF':>12} {'Rank BM25':>10}")
print("─" * 65)

tfidf_ranks = np.argsort(tfidf_scores)[::-1]
bm25_ranks  = np.argsort(bm25_norm)[::-1]

for i in range(len(chunk_texts)):
    tr = list(tfidf_ranks).index(i) + 1
    br = list(bm25_ranks).index(i) + 1
    print(f"{i:<8} {tfidf_scores[i]:>14.4f} {bm25_norm[i]:>13.4f} {tr:>12} {br:>10}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(chunk_texts))
width = 0.35

bars1 = ax.bar(x - width/2, tfidf_scores, width, label='TF-IDF', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, bm25_norm,    width, label='BM25 (normalized)', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Chunk Index')
ax.set_ylabel('Relevance Score')
ax.set_title(f'TF-IDF vs BM25 Scores\nQuery: "{query[:50]}..."')
ax.set_xticks(x)
ax.set_xticklabels([f'Chunk {i}' for i in range(len(chunk_texts))])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('tfidf_vs_bm25.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. End-to-End Mini RAG Pipeline <a id='5-rag'></a>

In [ ]:
class MiniRAGPipeline:
    """
    A minimal but complete Retrieval-Augmented Generation pipeline.
    
    Stages:
    1. Ingest     → load documents
    2. Preprocess → clean text
    3. Chunk      → split into pieces
    4. Index      → build TF-IDF + BM25 index
    5. Retrieve   → find relevant chunks for query
    6. Generate   → (placeholder — LLM step comes Day 5+)
    """
    
    def __init__(
        self,
        chunk_strategy: str = 'paragraph',
        chunk_size: int = 300,
        chunk_overlap: int = 50,
        top_k: int = 3
    ):
        self.chunk_strategy = chunk_strategy
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.top_k = top_k
        
        self.preprocessor = TextPreprocessor(remove_stops=False)  # Keep stops for context
        self.vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
        self.bm25 = BM25FromScratch()
        
        self.chunks = []
        self.tfidf_matrix = None
        self.is_fitted = False
    
    def _chunk(self, text: str) -> List[Dict]:
        """Apply selected chunking strategy."""
        if self.chunk_strategy == 'fixed':
            return fixed_size_chunk(text, self.chunk_size, self.chunk_overlap)
        elif self.chunk_strategy == 'sentence':
            return sentence_chunk(text, sentences_per_chunk=3)
        elif self.chunk_strategy == 'recursive':
            return recursive_chunk(text, self.chunk_size, self.chunk_overlap)
        else:  # paragraph (default)
            return paragraph_chunk(text)
    
    def ingest(self, documents: List[str], doc_names: List[str] = None):
        """Ingest, preprocess, and index a list of documents."""
        print(f"📥 Ingesting {len(documents)} document(s)...")
        
        all_chunks = []
        for doc_id, doc in enumerate(documents):
            name = doc_names[doc_id] if doc_names else f"doc_{doc_id}"
            
            # Step 2: Preprocess
            clean_doc = self.preprocessor.process(doc)
            
            # Step 3: Chunk
            chunks = self._chunk(doc)  # chunk original for readability
            for chunk in chunks:
                chunk['doc_id'] = doc_id
                chunk['doc_name'] = name
                all_chunks.append(chunk)
        
        self.chunks = all_chunks
        chunk_texts = [c['text'] for c in self.chunks]
        
        # Step 4: Index
        print(f"🗂️  Indexing {len(self.chunks)} chunks...")
        self.tfidf_matrix = self.vectorizer.fit_transform(chunk_texts)
        self.bm25.fit(chunk_texts)
        self.is_fitted = True
        
        print(f"✅ Pipeline ready!")
        print(f"   Strategy  : {self.chunk_strategy}")
        print(f"   Chunks    : {len(self.chunks)}")
        print(f"   Vocabulary: {self.tfidf_matrix.shape[1]} terms")
        return self
    
    def retrieve(self, query: str, method: str = 'tfidf') -> List[Dict]:
        """Retrieve top-k chunks for a query."""
        if not self.is_fitted:
            raise RuntimeError("Call .ingest() first!")
        
        chunk_texts = [c['text'] for c in self.chunks]
        
        if method == 'bm25':
            scores = np.array(self.bm25.get_scores(query))
        else:  # tfidf
            q_vec = self.vectorizer.transform([query])
            scores = cosine_similarity(q_vec, self.tfidf_matrix)[0]
        
        top_indices = np.argsort(scores)[::-1][:self.top_k]
        
        results = []
        for rank, idx in enumerate(top_indices):
            if scores[idx] > 0:
                result = dict(self.chunks[idx])
                result['rank'] = rank + 1
                result['score'] = float(scores[idx])
                result['method'] = method
                results.append(result)
        
        return results
    
    def ask(self, query: str, method: str = 'tfidf') -> str:
        """Retrieve relevant context and build a prompt for an LLM."""
        results = self.retrieve(query, method)
        
        if not results:
            return "No relevant context found."
        
        context = "\n\n---\n".join([f"[Source: {r['doc_name']}]\n{r['text']}" for r in results])
        
        # This prompt would be sent to an LLM (Day 5+)
        prompt = f"""Use the following context to answer the question.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""
        return prompt


print("✅ MiniRAGPipeline class defined")

In [ ]:
# ─── Run the Pipeline ─────────────────────────────────────────────────────────

docs = [SAMPLE_DOC]
doc_names = ["AI_Introduction"]

rag = MiniRAGPipeline(chunk_strategy='paragraph', top_k=2)
rag.ingest(docs, doc_names)

print("\n" + "="*65)
print("RETRIEVAL DEMO")
print("="*65)

query = "How does RAG reduce hallucination in language models?"
print(f"\n🔍 Query: {query}")

print("\n📌 TF-IDF Results:")
for r in rag.retrieve(query, 'tfidf'):
    print(f"  Rank {r['rank']} | Score: {r['score']:.4f} | {r['text'][:120]}...")

print("\n📌 BM25 Results:")
for r in rag.retrieve(query, 'bm25'):
    print(f"  Rank {r['rank']} | Score: {r['score']:.4f} | {r['text'][:120]}...")

print("\n" + "─"*65)
print("📋 Generated Prompt (would be sent to LLM on Day 5+):")
print("─"*65)
print(rag.ask(query, 'tfidf'))

---
## 6. Visualization & Analysis <a id='6-viz'></a>

In [ ]:
# ─── Heatmap: TF-IDF scores across chunks and query terms ────────────────────

query_terms = ['machine learning', 'neural network', 'language processing', 
               'retrieval', 'preprocessing']

# Build score matrix: rows=queries, cols=chunks
score_matrix = np.zeros((len(query_terms), len(chunk_texts)))
for i, qt in enumerate(query_terms):
    q_vec = vectorizer.transform([qt])
    score_matrix[i] = cosine_similarity(q_vec, tfidf_matrix)[0]

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(score_matrix, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(chunk_texts)))
ax.set_xticklabels([f'Chunk\n{i}' for i in range(len(chunk_texts))], fontsize=10)
ax.set_yticks(range(len(query_terms)))
ax.set_yticklabels(query_terms, fontsize=10)
ax.set_title('TF-IDF Relevance Heatmap: Queries × Chunks', fontsize=13, fontweight='bold')
ax.set_xlabel('Document Chunks')
ax.set_ylabel('Query Terms')

# Add score annotations
for i in range(score_matrix.shape[0]):
    for j in range(score_matrix.shape[1]):
        val = score_matrix[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=9, color='black' if val < 0.3 else 'white')

plt.colorbar(im, label='Cosine Similarity Score')
plt.tight_layout()
plt.savefig('tfidf_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print("📊 Heatmap saved as 'tfidf_heatmap.png'")

In [ ]:
# ─── Token Reduction through Preprocessing ────────────────────────────────────

sample_texts = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning models are trained on large datasets.",
    "Natural language processing handles text and speech.",
    "RAG combines retrieval and generation for better answers.",
    "Text preprocessing involves cleaning and normalizing input.",
]

configs = [
    ('Raw', TextPreprocessor(lowercase=False, remove_html=False, remove_special=False, remove_stops=False, stem=False)),
    ('Lowercase', TextPreprocessor(remove_html=False, remove_special=False, remove_stops=False, stem=False)),
    ('No Special', TextPreprocessor(remove_html=False, remove_stops=False, stem=False)),
    ('No Stops', TextPreprocessor(stem=False)),
    ('Stemmed', TextPreprocessor(stem=True)),
]

# Compute average vocabulary size for each config
vocab_sizes = []
for label, pp in configs:
    all_tokens = []
    for t in sample_texts:
        all_tokens.extend(pp.process(t).split())
    vocab_sizes.append((label, len(set(all_tokens)), len(all_tokens)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Effect of Preprocessing on Vocabulary & Token Count', fontweight='bold')

labels = [v[0] for v in vocab_sizes]
unique = [v[1] for v in vocab_sizes]
total  = [v[2] for v in vocab_sizes]
colors_grad = ['#c0392b', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']

axes[0].bar(labels, unique, color=colors_grad, edgecolor='white')
axes[0].set_title('Unique Vocabulary Size')
axes[0].set_ylabel('Unique Terms')
for i, v in enumerate(unique):
    axes[0].text(i, v+0.3, str(v), ha='center', fontweight='bold')

axes[1].bar(labels, total, color=colors_grad, edgecolor='white')
axes[1].set_title('Total Token Count')
axes[1].set_ylabel('Total Tokens')
for i, v in enumerate(total):
    axes[1].text(i, v+0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('preprocessing_effect.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Summary & Day 5 Preview <a id='7-summary'></a>

In [ ]:
summary = """
╔══════════════════════════════════════════════════════════════════╗
║              DAY 4 — SUMMARY CHEAT SHEET                        ║
╠══════════════════════════════════════════════════════════════════╣
║  PREPROCESSING                                                   ║
║  ├─ Lowercase    → Normalize casing                              ║
║  ├─ Remove HTML  → Strip markup from web data                    ║
║  ├─ Remove Punct → Clean noise for keyword models                ║
║  ├─ Tokenize     → Split into words or sentences                 ║
║  ├─ Stop Words   → Remove for TF-IDF, KEEP for embeddings        ║
║  ├─ Stemming     → Fast but crude (may produce non-words)        ║
║  └─ Lemmatize    → Slower but accurate (valid base forms)        ║
╠══════════════════════════════════════════════════════════════════╣
║  CHUNKING                                                        ║
║  ├─ Fixed-Size   → Simple, fast, may break sentences             ║
║  ├─ Sentence     → Natural breaks, variable size                 ║
║  ├─ Paragraph    → High coherence, very variable size            ║
║  ├─ Recursive    → Best general-purpose (LangChain default)      ║
║  └─ Semantic     → Best quality, needs embeddings (Day 5!)       ║
╠══════════════════════════════════════════════════════════════════╣
║  RETRIEVAL                                                       ║
║  ├─ TF-IDF       → Term importance × rarity, cosine similarity  ║
║  ├─ BM25         → TF-IDF + saturation + length norm            ║
║  └─ Cosine Sim   → Angle between vectors (0=diff, 1=same)       ║
╠══════════════════════════════════════════════════════════════════╣
║  RAG PIPELINE                                                    ║
║  Ingest → Preprocess → Chunk → Index → Retrieve → Generate      ║
╠══════════════════════════════════════════════════════════════════╣
║  COMING TOMORROW (Day 5):                                        ║
║  • Dense embeddings (word2vec intuition)                         ║
║  • Semantic similarity vs keyword matching                        ║
║  • Preparing chunks for a real embedding-based RAG pipeline      ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(summary)